In [12]:
import pandas as pd 
import numpy as np 
import glob 

import joblib

from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import ( 
    confusion_matrix, 
    classification_report, 
    accuracy_score 
    ) 

import tensorflow as tf

In [13]:
model = tf.keras.models.load_model("room_classifier.h5") 
print("Model loaded successfully")

Model loaded successfully


In [14]:
csv_files = glob.glob("*.csv") 
dfs = [] 

for file in csv_files: 
    try: 
        temp_df = pd.read_csv(file) 
        dfs.append(temp_df) 
    except Exception as e: 
        print(f"Error loading {file}: {e}") 
        
df = pd.concat(dfs, ignore_index=True) 
df = df.dropna() 

df["sound"] = pd.to_numeric(df["sound"], errors="coerce") 
df["light"] = pd.to_numeric(df["light"], errors="coerce") 

df = df.dropna() 

print("Rows:", len(df)) 
print(df.head())

Rows: 613
   sound  light label
0  10.19    3.0  dark
1  14.15    3.0  dark
2  22.99    3.0  dark
3  17.93    3.0  dark
4  21.81    3.0  dark


In [15]:
encoder = LabelEncoder() 

df["label"] = encoder.fit_transform(df["label"]) 

print("Label Mapping:") 

for i, label in enumerate(encoder.classes_): 
    print(f"{label} -> {i}")

Label Mapping:
dark -> 0
focus -> 1
noisy -> 2


In [16]:
X = df[["sound", "light"]].astype("float32") 
y = df["label"].astype("int32")

scaler = joblib.load("scaler.pkl")

X_scaled = scaler.transform(X)

In [17]:
predictions = model.predict(X_scaled) 
predicted_labels = np.argmax(predictions, axis=1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [18]:
accuracy = accuracy_score(y, predicted_labels) 
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 96.9 %


In [19]:
cm = confusion_matrix(y, predicted_labels) 
print(cm)

[[131   0   7]
 [  0 152   3]
 [  9   0 311]]


In [20]:
print( 
    classification_report( 
        y, 
        predicted_labels, 
        target_names=encoder.classes_ 
    ) 
)

              precision    recall  f1-score   support

        dark       0.94      0.95      0.94       138
       focus       1.00      0.98      0.99       155
       noisy       0.97      0.97      0.97       320

    accuracy                           0.97       613
   macro avg       0.97      0.97      0.97       613
weighted avg       0.97      0.97      0.97       613



In [22]:
sound = 12.0 
light = 45.0 

sample = np.array([[sound, light]], dtype=np.float32) 

sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled) 

class_id = np.argmax(prediction) 

label = encoder.inverse_transform([class_id])[0] 

print("Prediction:", label) 
print("Probabilities:", prediction[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction: dark
Probabilities: [0.67911875 0.15011448 0.17076674]


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [23]:
print(encoder.classes_)
print("Means:", scaler.mean_)
print("Stds:", scaler.scale_)

['dark' 'focus' 'noisy']
Means: [ 47.89593862 153.47755102]
Stds: [ 60.26405792 133.07877254]


In [24]:
print(model.input_shape)
print(model.output_shape)

(None, 2)
(None, 3)
